In [ ]:
import kagglehub

path = kagglehub.dataset_download(
    "uciml/electric-power-consumption-data-set"
)

print("Dataset Path:", path)

In [ ]:
import os

for file in os.listdir(path):
    print(file)

### **Import Dataset**

In [ ]:
import pandas as pd
import os

file_path = os.path.join(
    path,
    "household_power_consumption.txt"
)

df = pd.read_csv(
    file_path,
    sep=';',
    low_memory=False
)

df.head()

### **Understand the Data**

In [ ]:
df.info()

df.isnull().sum()

### **Data Cleaning**

In [ ]:
df['Datetime'] = pd.to_datetime(
    df['Date'] + ' ' + df['Time'],
    dayfirst=True
)

df.set_index('Datetime', inplace=True)

df.head()

### **Selecting the Forecast Value**

In [ ]:
df['Global_active_power'] = pd.to_numeric(
    df['Global_active_power'],
    errors='coerce'
)

power = df['Global_active_power']

### **Handling the missing Value**

In [ ]:
power = power.fillna(method='ffill')

### **Resampling the Data**

In [ ]:
daily_power = power.resample('D').mean()

daily_power.head()

### **Exploratory Analysis**

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(15,6))

plt.plot(daily_power)

plt.title("Daily Electricity Consumption")
plt.xlabel("Date")
plt.ylabel("Power")

plt.show()

### **Time Series Decomposition**

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

decomposition = seasonal_decompose(
    daily_power,
    model='additive',
    period=365
)

decomposition.plot()
plt.show()

### **Train Test Split**

In [ ]:
train = daily_power[:-365]

test = daily_power[-365:]

print(len(train))
print(len(test))

### **Holt-Winters Forecasting**

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

model = ExponentialSmoothing(
    train,
    trend='add',
    seasonal='add',
    seasonal_periods=365
).fit()

### **Generate The Forecast**

In [ ]:
forecast = model.forecast(365)

forecast.head()

### **Comparing the Results**

In [ ]:
plt.figure(figsize=(15,6))

plt.plot(test.index, test, label='Actual')

plt.plot(
    forecast.index,
    forecast,
    label='Forecast'
)

plt.legend()

plt.title("Actual vs Forecast")

plt.show()

### **Evalute the Forecast**

In [ ]:
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error

mae = mean_absolute_error(test, forecast)

mse = mean_squared_error(test, forecast)

rmse = mse ** 0.5

print("MAE :", mae)
print("RMSE :", rmse)

In [ ]:
print(daily_power.min())
print(daily_power.max())
print(daily_power.mean())